In [134]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from data.pull_data import pull_lobster_data

import pandas as pd
import numpy as np

VOL_THRESHOLD = 5
VOL_TARGET = 0.15
SECONDS_PER_MINUTE = 60
SECONDS_PER_HOUR = 3600
SECONDS_PER_DAY = 23400
SECONDS_PER_YEAR = SECONDS_PER_DAY * 252
HALFLIFE_WINSORISE = SECONDS_PER_DAY * 60

from typing import List, Tuple

In [135]:
df = pull_lobster_data('AAPL')

df = df[
    ~df['mid_price'].isna()
    | ~df['mid_price'].isnull()
    | (df['mid_price'] > 1e-8)  # price is zero
].copy()



# winsorize using rolling 5X standard deviations to remove extreme outliers
df['srs'] = df['mid_price']
ewm = df['srs'].ewm(halflife=HALFLIFE_WINSORISE) # TODO: decide on halflife when frequency is 's'
avgs = ewm.mean()
stds = ewm.std()
df['srs'] = np.clip(df['srs'], avgs + VOL_THRESHOLD * stds, avgs - VOL_THRESHOLD * stds)

In [136]:
def calc_returns(srs: pd.Series, offset: int = 1) -> pd.Series:
    """Calculates returns over the specified number of seconds."""
    return srs / srs.shift(offset) - 1.0

def calc_vol(returns: pd.Series, span: int = SECONDS_PER_DAY * 5) -> pd.Series:
    """Calculates exponentially weighted volatility over given span (default ~5 trading days)."""
    return (
        returns.ewm(span=span, min_periods=span)
        .std()
        .bfill()
    )

def calc_vol_scaled_returns(returns: pd.Series, vol: pd.Series = pd.Series(dtype=float)) -> pd.Series:
    """Scales returns to target volatility using annualised volatility."""
    if vol.empty:
        vol = calc_vol(returns)
    annualised_vol = vol * np.sqrt(SECONDS_PER_YEAR)
    return returns * VOL_TARGET / annualised_vol.shift(1)

df_asset = df.copy()

# Apply functions to your DataFrame
df_asset['second_returns'] = calc_returns(df_asset['srs'], offset=1)
df_asset['second_vol'] = calc_vol(df_asset['second_returns'])
df_asset['target_returns'] = calc_vol_scaled_returns(df_asset['second_returns'], df_asset['second_vol']).shift(-1)

# Normalised returns for different time windows
def calc_normalised_returns(offset_seconds: int) -> pd.Series:  # docstring left as is
    returns = calc_returns(df_asset['srs'], offset=offset_seconds)
    return returns / df_asset['second_vol'] / np.sqrt(offset_seconds)

# Add normalised return columns
df_asset['norm_second_return'] = calc_normalised_returns(1)
df_asset['norm_minute_return'] = calc_normalised_returns(SECONDS_PER_MINUTE)
df_asset['norm_hour_return'] = calc_normalised_returns(SECONDS_PER_HOUR)
df_asset['norm_day_return'] = calc_normalised_returns(SECONDS_PER_DAY)
df_asset['norm_week_return'] = calc_normalised_returns(SECONDS_PER_DAY * 5)
df_asset['norm_month_return'] = calc_normalised_returns(SECONDS_PER_DAY * 21)
df_asset['norm_quarter_return'] = calc_normalised_returns(SECONDS_PER_DAY * 63)
# df_asset['norm_annual_return'] = calc_normalised_returns(SECONDS_PER_DAY * 252)

In [137]:
class MACDStrategy:
    def __init__(self, trend_combinations: List[Tuple[float, float]] = None):
        if trend_combinations is None:
            self.trend_combinations = [
                (300, 900),  # 5 min vs 15 min
                (600, 1800),  # 10 min vs 30 min
                (1800, 7200),  # 30 min vs 2 hours
                (3600, 14400),  # 1 hr vs 4 hrs
                (7200, 28800),  # 2 hrs vs 8 hrs
                (14400, 86400)  # 4 hrs vs 24 hrs
            ]
        else:
            self.trend_combinations = trend_combinations

    @staticmethod
    def calc_signal(srs: pd.Series, short_timescale: int, long_timescale: int) -> float:
        def _calc_halflife(timescale):
            return np.log(0.5) / np.log(1 - 1 / timescale)

        short_ema = srs.ewm(halflife=_calc_halflife(short_timescale)).mean()
        long_ema = srs.ewm(halflife=_calc_halflife(long_timescale)).mean()
        macd = short_ema - long_ema

        # Rolling std over e.g. 4 hours
        window_seconds = 14_400
        rolling_std = srs.rolling(window_seconds).std().bfill()

        # Annualisation factor
        SECONDS_PER_YEAR = 252 * 23_400
        annualisation_factor = np.sqrt(SECONDS_PER_YEAR / window_seconds)

        q = macd / rolling_std
        q = q * annualisation_factor

        return q

    @staticmethod
    def scale_signal(y):
        return y * np.exp(-(y ** 2) / 4) / 0.89

    def calc_combined_signal(self, srs: pd.Series) -> float:
        return np.sum(
            [self.calc_signal(srs, S, L) for S, L in self.trend_combinations]
        ) / len(self.trend_combinations)


In [138]:
trend_combinations = [
    (300, 900),  # 5 min vs 15 min
    (600, 1800),  # 10 min vs 30 min
    (1800, 7200),  # 30 min vs 2 hours
    (3600, 14400),  # 1 hr vs 4 hrs
    (7200, 28800),  # 2 hrs vs 8 hrs
    (14400, 86400)  # 4 hrs vs 24 hrs
]

for short_window, long_window in trend_combinations:
        df_asset[f'macd_{short_window}_{long_window}'] = MACDStrategy.calc_signal(
            df_asset['srs'], short_window, long_window
        )

In [139]:
if len(df_asset):
    df_asset['hour_of_day'] = df_asset.index.hour  # NEW
    df_asset['minute_of_hour'] = df_asset.index.minute  # NEW
    df_asset['second_of_minute'] = df_asset.index.second  # NEW
    df_asset['day_of_week'] = df_asset.index.dayofweek
    df_asset['day_of_month'] = df_asset.index.day
    df_asset['week_of_year'] = df_asset.index.isocalendar().week
    df_asset['month_of_year'] = df_asset.index.month
    df_asset['year'] = df_asset.index.year
    df_asset['date'] = df_asset.index  # duplication but sometimes makes life easier
else:
    df_asset['hour_of_day'] = []
    df_asset['minute_of_hour'] = []
    df_asset['second_of_minute'] = []
    df_asset['day_of_week'] = []
    df_asset['day_of_month'] = []
    df_asset['week_of_year'] = []
    df_asset['month_of_year'] = []
    df_asset['year'] = []
    df_asset['date'] = []


In [ ]:
features = pd.concat(
        [
            deep_momentum_strategy_features(pull_quandl_sample_data(ticker)).assign(
                ticker=ticker
            )
            for ticker in tickers
        ]
    )

,mid_price,srs,second_returns,second_vol,target_returns,norm_second_return,norm_minute_return,norm_hour_return,norm_day_return,norm_week_return,...,macd_14400_86400,hour_of_day,minute_of_hour,second_of_minute,day_of_week,day_of_month,week_of_year,month_of_year,year,date
date,,,,,,,,,,,,,,,,,,,,,
2024-04-04 10:30:00,171.015,141.455811,-1.027987e-07,5.843680e-08,-1.086631e-04,-1.759143,-13.463485,-116.283682,-441.205720,-956.225660,...,-457.772767,10,30,0,3,4,14,4,2024,2024-04-04 10:30:00
2024-04-04 10:30:01,171.015,141.455796,-1.027982e-07,5.843703e-08,-1.086622e-04,-1.759128,-13.472417,-116.278004,-441.190906,-956.216207,...,-457.785879,10,30,1,3,4,14,4,2024,2024-04-04 10:30:01
2024-04-04 10:30:02,171.015,141.455782,-1.027978e-07,5.843725e-08,-1.086613e-04,-1.759113,-13.481349,-116.271920,-441.176093,-956.206755,...,-457.798989,10,30,2,3,4,14,4,2024,2024-04-04 10:30:02
2024-04-04 10:30:03,171.015,141.455767,-1.027973e-07,5.843748e-08,-1.086604e-04,-1.759099,-13.487275,-116.265768,-441.161280,-956.197303,...,-457.812097,10,30,3,3,4,14,4,2024,2024-04-04 10:30:03
2024-04-04 10:30:04,171.015,141.455752,-1.027968e-07,5.843770e-08,-1.086595e-04,-1.759084,-13.492196,-116.257220,-441.146467,-956.187879,...,-457.825202,10,30,4,3,4,14,4,2024,2024-04-04 10:30:04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31 15:59:54,250.550,114.248666,7.839049e-10,9.432385e-08,1.126001e-06,0.008311,0.325973,1.185008,-18.129765,-526.923223,...,-5848.004191,15,59,54,1,31,1,12,2024,2024-12-31 15:59:54
2024-12-31 15:59:55,250.510,114.248666,1.719401e-09,9.432399e-08,1.890182e-06,0.018229,0.316321,1.184686,-18.126163,-526.916746,...,-5847.597619,15,59,55,1,31,1,12,2024,2024-12-31 15:59:55
2024-12-31 15:59:56,250.460,114.248666,2.886308e-09,9.432416e-08,8.970185e-07,0.030600,0.309055,1.184674,-18.122520,-526.910112,...,-5847.191201,15,59,56,1,31,1,12,2024,2024-12-31 15:59:56


In [ ]:
features = pd.concat(
        [
            deep_momentum_strategy_features(pull_quandl_sample_data(ticker)).assign(
                ticker=ticker
            )
            for ticker in tickers
        ]
    )

In [ ]:


def main(
    tickers: List[str],
    cpd_module_folder: str,
    lookback_window_length: int,
    output_file_path: str,
    extra_lbw: List[int],
):
    features = pd.concat(
        [
            deep_momentum_strategy_features(pull_quandl_sample_data(ticker)).assign(
                ticker=ticker
            )
            for ticker in tickers
        ]
    )

    features.date = features.index
    features.index.name = "Date"

    if lookback_window_length:
        features_w_cpd = include_changepoint_features(
            features, cpd_module_folder, lookback_window_length
        )

        if extra_lbw:
            for extra in extra_lbw:
                extra_data = pd.read_csv(
                    output_file_path.replace(
                        f"quandl_cpd_{lookback_window_length}lbw.csv",
                        f"quandl_cpd_{extra}lbw.csv",
                    ),
                    index_col=0,
                    parse_dates=True,
                ).reset_index()[
                    ["date", "ticker", f"cp_rl_{extra}", f"cp_score_{extra}"]
                ]
                extra_data["date"] = pd.to_datetime(extra_data["date"])

                features_w_cpd = pd.merge(
                    features_w_cpd.set_index(["date", "ticker"]),
                    extra_data.set_index(["date", "ticker"]),
                    left_index=True,
                    right_index=True,
                ).reset_index()
                features_w_cpd.index = features_w_cpd["date"]
                features_w_cpd.index.name = "Date"
        else:
            features_w_cpd.index.name = "Date"
        features_w_cpd.to_csv(output_file_path)
    else:
        features.to_csv(output_file_path)


if __name__ == "__main__":

    def get_args():
        """Returns settings from command line."""

        parser = argparse.ArgumentParser(description="Run changepoint detection module")
        # parser.add_argument(
        #     "cpd_module_folder",
        #     metavar="c",
        #     type=str,
        #     nargs="?",
        #     default=CPD_QUANDL_OUTPUT_FOLDER_DEFAULT,
        #     # choices=[],
        #     help="Input folder for CPD outputs.",
        # )
        parser.add_argument(
            "lookback_window_length",
            metavar="l",
            type=int,
            nargs="?",
            default=None,
            # choices=[],
            help="Input folder for CPD outputs.",
        )
        # parser.add_argument(
        #     "output_file_path",
        #     metavar="f",
        #     type=str,
        #     nargs="?",
        #     default=FEATURES_QUANDL_FILE_PATH_DEFAULT,
        #     # choices=[],
        #     help="Output file location for csv.",
        # )
        parser.add_argument(
            "extra_lbw",
            metavar="-e",
            type=int,
            nargs="*",
            default=[],
            # choices=[],
            help="Fill missing prices.",
        )

        args = parser.parse_known_args()[0]

        return (
            QUANDL_TICKERS,
            CPD_QUANDL_OUTPUT_FOLDER(args.lookback_window_length),
            args.lookback_window_length,
            FEATURES_QUANDL_FILE_PATH(args.lookback_window_length),
            args.extra_lbw,
        )

    main(*get_args())
